## [Tutorial](https://github.com/biohub/esm/tree/main/cookbook/tutorials): How to run minibinder + scFv design fully end-to-end.

In this notebook we will use [Modal](https://modal.com/) to parallelize binder design and synthesize a selection, using the protocol described in the ESMC and ESMFold2 paper titled ["Language Modeling Materializes a World Model of Protein Biology"](https://biohub.ai/papers/esm_protein.pdf).

Biohub used this approach to design minibinders and scFvs against five therapeutically relevant targets — PDGFRB, EGFR, PD-L1, CD45, and CTLA4 — spanning receptor tyrosine kinases, immune checkpoints, and cell-surface phosphatases. Binders exhibit nanomolar affinity, target specificity, and functional activity in laboratory assays.


**You'll need:**
- A target protein sequence (or pick one of the built-in presets)
- A [Modal](https://modal.com/) account and token (free tier works to get started)
- The notebook contains cells launching independent trajectories and a small sweep (N=256). With Modal H100 pricing, this will cost ≈$2 and ≈$150, respectively

**You'll get:** a file of top-ranked designed binder sequences, plus 3D structures of the predicted complexes that you can view in the notebook.

**Workflow:**
1. **Setup** (one-time): install dependencies, get a Modal token, deploy the design app
2. **Try one job**: pick a target and binder type, run a single design end-to-end as a sanity check
3. **Run a sweep**: launch many parallel jobs to produce real candidates
4. **Pick the designs to order**: filter and rank, save the shortlist


> **Why does this notebook need Modal?** Each design job needs a GPU and a few minutes of compute, and a real campaign means launching hundreds of these in parallel. [Modal](https://modal.com/) is a cloud service that lets this notebook spin up remote GPUs on demand, run each job on one, and return results to you. You don't manage any servers or containers yourself. When you "deploy" the design app to Modal (in section 1), you're uploading a Python file that defines the job; Modal then runs that code for you whenever the notebook calls `app.design.spawn(...)`.

### 1. Setup

In [5]:
# Environment
! pip install esm@git+https://github.com/Biohub/esm.git@main
! pip install modal py3dmol pyarrow

  Cloning https://github.com/Biohub/esm.git (to revision main) to /tmp/pip-install-_3fz1zof/esm_d3f8a9fbb6714c0381ac40c7b01011ce
  Running command git clone --filter=blob:none --quiet https://github.com/Biohub/esm.git /tmp/pip-install-_3fz1zof/esm_d3f8a9fbb6714c0381ac40c7b01011ce
  Resolved https://github.com/Biohub/esm.git to commit eca8e1cd2c502755670b24177311cb614ec2ba85
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/Biohub/transformers.git (to revision main) to /tmp/pip-install-_3fz1zof/transformers_3e4f207119844d858b6e16879b12587f
  Running command git clone --filter=blob:none --quiet https://github.com/Biohub/transformers.git /tmp/pip-install-_3fz1zof/transformers_3e4f207119844d858b6e16879b12587f
  Resolved https://github.com/Biohub/transformers.git to commit ef32577f55da19a4989cd7b22e004dc43a4998cb
  Installing build dependencies ... done
  Getting requirements to

In [ ]:
! #copy api token

In [7]:
# Confirm you have a modal token, or make one
! modal token info  # Check
# ! modal token new  # Create

Token: ak-aQgOz9Vf5VBMAcCoFWONEx
Name: s100b
Workspace: tveshaghosh (ac-0Y1BkphidhGNzOQSm2VODd)
User: tveshaghosh (us-KEUMRgtfSBgwiqHEeV4ckR)
Created at: 2026-06-09 12:23:07 UTC


### Deploy the design app to Modal

The file `binder_design.py` lives in the same folder as this notebook. It defines the GPU job, the model, and the design loop.

You only need to deploy once. Re-run this cell only if the underlying `.py` file changes.

### If you're running on Colab

Colab only pulls the notebook itself when you open it, not the surrounding files from the repo. Run the cell below to grab `binder_design.py` into your Colab workspace. If you're running locally, skip it.

In [8]:
# Colab only: download binder_design.py into the working directory
! wget -q https://raw.githubusercontent.com/Biohub/esm/main/cookbook/tutorials/binder_design.py
! ls binder_design.py

binder_design.py


In [9]:
# Deploy (or redeploy after changing binder_design.py).
# This only needs to be run a single time, unless code in binder_design.py changes.
! modal deploy binder_design.py

2026-06-09 16:05:55.727860: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-09 16:05:56.372509: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
ESMC: transformer_engine is not installed; falling back to pure-PyTorch LayerNorm+Linear / LayerNorm+MLP. Outputs will differ numerically — measured on the unnormalized residual stream (before the final LayerNorm), ~O(10) max-diff in fp32 and ~O(100) in bf16; after the final LayerNorm these shrink to a few ULP and perplexity stays within rounding noise. Install with `pip install transformer-engine[pytorch]` to enable fused fp32-reduction LayerNorm.
ESMC: neither xformers nor flash-attn is installed; falling back to PyTorch ``F.scaled_

### Imports

In [10]:
from itertools import product
from pathlib import Path

import modal
import pandas as pd
import py3Dmol
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from tqdm.auto import tqdm

### App setup

`modal.Cls.from_name(...)` grabs a handle to the design app you just deployed, without rerunning anything on Modal yet. Instantiating it gives you `app`, which is what you'll call `app.design.spawn(...)` on to launch design jobs.

`use_scaling_critics=False` is the default. Setting it to `True` adds extra critic models from the paper that improve selection at the cost of more compute per job.

In [11]:
ESMFold2Design = modal.Cls.from_name("esmfold2-design", "ESMFold2DesignModal")
# Set 'use_scaling_critics' to evaluate with the additional critics.
# Off by default. But cells below were populated with them enabled.
app = ESMFold2Design(use_scaling_critics=False)

## 2. Try one design job

Run a single job end-to-end before launching a sweep. This is a sanity check that everything is wired up and that the target/scaffold combo you've chosen produces a sensible complex.

Pick **one** of the two options below and run only that cell. (They both define a variable called `future`, so running both back-to-back overwrites the first one.)

**Option 1** uses a built-in target and binder scaffold. Available targets: `ctla4`, `egfr`, `pdgfrb`, `pd-l1`, `cd45`. Available binder types: `minibinder`, `trastuzumab_framework_vhvl` (an antibody scaffold). Easiest if your target is one of the built-ins.

**Option 2** takes your own target sequence and binder scaffold. In the binder scaffold, `#` means "design this position" and any amino acid letter means "keep this position fixed." For example:
- `"#" * 60` designs a fully free 60-residue minibinder
- A trastuzumab-style antibody scaffold (shown below) fixes the framework regions and lets the model design the CDR loops

If you're designing an antibody, pass `is_antibody=True` so the selection step later uses the antibody-appropriate scoring.

Jobs run on Modal in the background. The `dashboard_url` link the cell prints is a clickable link to live progress.

In [ ]:
# ---- Option 1: Use presets. ----
# Relies on the registry in modal_binder_design.py::{TARGET_SEQUENCES,BINDER_PROMPT_FACTORIES}, which can be modified.
# future = app.design.spawn(target_name="ctla4", binder_name="minibinder")
# future.get_dashboard_url()  # A clickable link to Modal dashboard

'https://modal.com/id/fc-01KT1ZA3NQ0JTF2B4HNCR159NM'

In [12]:
# ---- Option 2: Provide your own sequences. ----
# Our s100b sequence crop.
# ---- Option 2: Provide your own sequences. ----
s100b_sequence = "SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEEIKEQEVVDKVMETLDNDGDGECDFQEFMAFVAMVTTACHEFFEHE"

future = app.design.spawn(
    target_sequence=s100b_sequence,
    binder_sequence="#" * 50,   # 60 fully-designed residues
)
future.get_dashboard_url() #clickable link to modal dashboard

ResourceExhaustedError: Function call failed: workspace billing cycle spend limit reached

In [9]:
# ---- Monitor ----
# Tail a function's output here in jupyter
! modal app logs esmfold2-design -f --function-call {future.object_id}

Loading CCD dictionary from /models/hub/models--biohub--ESMFold2/snapshots/1ebf0e3481a5184eb6171d40615c79e384b48796/ccd.pkl
2026-06-09 12:43:43,166 INFO:   step   0  |  intra_contact_loss=4.3593  inter_contact_loss=4.2467  glob_loss=4.6160  total_loss=8.3512  plm_loss=3.1250  T=0.9999
2026-06-09 12:43:46,344 INFO:   step   5  |  intra_contact_loss=4.2587  inter_contact_loss=3.9739  glob_loss=4.3006  total_loss=8.3202  plm_loss=3.3438  T=0.9961
2026-06-09 12:43:49,119 INFO:   step  10  |  intra_contact_loss=4.2525  inter_contact_loss=3.8379  glob_loss=4.1631  total_loss=8.1435  plm_loss=3.2656  T=0.9869
2026-06-09 12:43:51,741 INFO:   step  15  |  intra_contact_loss=4.1749  inter_contact_loss=3.5494  glob_loss=3.8151  total_loss=7.9377  plm_loss=3.3125  T=0.9725
2026-06-09 12:43:54,353 INFO:   step  20  |  intra_contact_loss=3.8375  inter_contact_loss=3.9050  glob_loss=2.8977  total_loss=7.7633  plm_loss=3.3125  T=0.9529
2026-06-09 12:43:57,164 INFO:   step  25  |  intra_contact_loss=4.

In [11]:
# ---- Load result ----
best_sequences, trajectory, critic_results = future.get()
print("Best sequences: ", best_sequences)
df = pd.DataFrame(critic_results)
df.drop(columns=["logits", "complex"])

Best sequences:  ['SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEEIKEQEVVDKVMETLDNDGDGECDFQEFMAFVAMVTTACHEFFEHE|DVDVVVLVAVLFANFAVNNFGGGGAAAGRIAAAAASLAAALMKRAGIAAF']


,is_antibody,critic_name,batch_idx,designed_sequence,final_loss,iptm,distogram_iptm_proxy,cdr_distogram_iptm_proxy
0,False,ESMFold2-Experimental-Fast,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,6.050306,0.234571,0.384033,NaN
1,False,ESMFold2-Experimental-Fast-Cutoff2025,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,6.050306,0.248579,0.399929,NaN
2,False,ESMFold2-Experimental,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,6.050306,0.200595,0.399929,NaN
3,False,ESMFold2-Experimental-Cutoff2025,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,6.050306,0.222055,0.372111,NaN


In [12]:
# ---- Visualize ----
protein_complex = (
    df[df.critic_name.eq("ESMFold2-Experimental-Cutoff2025")].iloc[0].complex
)
(
    py3Dmol.view(width=600, height=600)
    .addModel(protein_complex.to_pdb_string(), "pdb")
    .setStyle({"chain": "A"}, {"cartoon": {"color": "green"}})  # pyright: ignore
    .setStyle({"chain": "B"}, {"cartoon": {"color": "cyan"}})  # pyright: ignore
    .addStyle(  # pyright: ignore
        {"not": {"atom": ["N", "C", "O"]}},
        {"stick": {"colorscheme": "default", "radius": 0.2}},
    )
    .center()  # pyright: ignore
    .zoomTo()  # pyright: ignore
)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 3. Run a sweep for real designs
For real candidates worth ordering, sweep across many seeds (and optionally multiple targets, binder types, or lengths) and select the best.

Edit `line_sweeps` below to define your campaign. Each key is a sweep axis; the notebook runs one job per combination of values. The default below sweeps 128 seeds across two binder types against PD-L1.

**Before you click Run on the Launch cell, check the printed shape of the dataframe and confirm it's the number of jobs you intended.**

In [14]:
# ---- Config ----
save_dir = Path("sweep")
save_dir.mkdir(exist_ok=True)

s100b_sequence = "SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEEIKEQEVVDKVMETLDNDGDGECDFQEFMAFVAMVTTACHEFFEHE"

line_sweeps = dict(
    target_name=[None],
    target_sequence=[s100b_sequence],
    binder_name=[None],
    binder_sequence=["#" * 60],
    use_scaling_critics=[True],
    seed=list(range(50)),        # 50 seeds → 50 designs
    batch_size=[1],
)
df = pd.DataFrame(product(*line_sweeps.values()), columns=list(line_sweeps.keys()))
display(df.head(2))
df.shape

,target_name,target_sequence,binder_name,binder_sequence,use_scaling_critics,seed,batch_size
0,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,##############################################...,True,0,1
1,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,##############################################...,True,1,1


(50, 7)

In [15]:
# ---- Launch ----
df["call_id"] = [
    app.design.spawn(
        target_name=row.target_name,
        target_sequence=row.target_sequence,
        binder_name=row.binder_name,
        binder_sequence=row.binder_sequence,
        seed=row.seed,
        batch_size=row.batch_size,
    ).object_id
    for row in df.itertuples()
]
df.to_parquet(save_dir / "manifest.parquet", index=False)
print(
    f"Spawned {len(df)} jobs. It is safe to close the notebook."
    "The next cell will resume from call_id's, saved by Modal for up to 7 days."
)

Spawned 50 jobs. It is safe to close the notebook.The next cell will resume from call_id's, saved by Modal for up to 7 days.


### Coming back later (important)

Your jobs are now running on Modal's GPUs. **You do not need to keep this notebook open.** Modal continues running the jobs on its own and holds the results for up to 7 days.

**If you're running on Colab:** the runtime is wiped when you disconnect. To resume a sweep on Colab, mount Google Drive **before** running the Launch cell and point `save_dir` at a Drive path, e.g.:

\`\`\`python
from google.colab import drive
drive.mount('/content/drive')
save_dir = Path('/content/drive/MyDrive/binder_sweep')
\`\`\`

When you reopen the notebook later, you'll need to re-install the dependencies, re-authenticate Modal (`modal token new`), re-mount Drive, then resume from the Monitor cell.

In [3]:
# ---- Monitor ----
from tqdm.contrib.concurrent import thread_map

df = pd.read_parquet(save_dir / "manifest.parquet")
df["future"] = thread_map(modal.FunctionCall.from_id, df.call_id.values)
df["status"] = thread_map(lambda f: f.get_call_graph()[0].status.name, df.future.values)
print("First task url: ", df.at[0, "future"].get_dashboard_url())  # pyright: ignore
df.status.value_counts()

NameError: name 'pd' is not defined

The Collect cell waits for all jobs that succeeded and unpacks their results. Jobs that failed or are still running are skipped, so you can re-run Monitor + Collect periodically as more jobs finish.

In [2]:
# ---- Collect ----
df_success = df[df.status.eq("SUCCESS")].copy()
df_success["result"] = thread_map(
    lambda x: x.get(), df_success.future.values, max_workers=32, chunksize=32
)  # Blocks until all jobs are complete.
df_success["result_df"] = [pd.DataFrame(r[2]) for r in tqdm(df_success.result)]  # pyright: ignore

NameError: name 'df' is not defined

## 4. Pick the designs to order

This is your final shortlist. The cell below:

1. Combines results from all successful jobs into one dataframe.
2. Filters minibinders to isoelectric point under 6 (helps with solubility and expression). Antibodies pass through unfiltered.
3. Scores each unique designed sequence by averaging `iptm` (interface predicted TM-score, higher is better) and an `iptm_proxy` term across its trajectories.
4. Returns the top 84 designs per (target, binder type), saved to `selection.parquet` inside your `save_dir`.

84 is a plate-friendly number for ordering and screening. The cutoff and the isoelectric-point filter are currently hardcoded inside the cell, so to change them, edit the values directly in the function.


In [30]:
# ---- Select (instrumented) ----

# Tunables ----------------------------------------------------------
PI_THRESHOLD = None      # minibinders must have isoelectric point below this. Raise (e.g. 8.0) or set to None to disable.
TOP_N = 84              # designs to keep per (target, binder) group
SCALING_CHECKPOINT_SUBSTRING = "ESMFold2-Experimental-Fast-base"
# -------------------------------------------------------------------

# Join all result_df's, with other fields in df broadcasted as metadata.
df_result = pd.concat(
    [
        row.result_df.assign(**row.drop(["result", "result_df"]).to_dict())
        for _, row in df_success.iterrows()
    ],
    ignore_index=True,
    axis=0,
)
print(f"[1] rows collected from successful jobs: {len(df_result)}")
print(f"    unique designed_sequence: {df_result.designed_sequence.nunique()}")

if df_result.empty:
    raise ValueError(
        "df_result is empty — no successful job results were collected. "
        "Check df.status.value_counts() and len(df_success) upstream; the "
        "problem is in Monitor/Collect, not here."
    )

# Compute isoelectric point of the binder chain (text after the '|').
df_result["binder_sequence"] = df_result.designed_sequence.str.split(r"\|").str[1]
df_result["isoelectric_point"] = [
    ProteinAnalysis(seq).isoelectric_point()
    for seq in tqdm(df_result.binder_sequence.values)
]
print(
    f"[2] pI distribution: min={df_result.isoelectric_point.min():.2f} "
    f"median={df_result.isoelectric_point.median():.2f} "
    f"max={df_result.isoelectric_point.max():.2f}"
)

# Isoelectric point filter (antibodies always pass; minibinders need pI < threshold).
if PI_THRESHOLD is None:
    df_filter = df_result.copy()
    print("[3] pI filter DISABLED — keeping all designs")
else:
    df_filter = df_result[
        df_result.is_antibody | df_result.isoelectric_point.lt(PI_THRESHOLD)
    ]
    n_drop = df_result.designed_sequence.nunique() - df_filter.designed_sequence.nunique()
    print(
        f"[3] after pI<{PI_THRESHOLD} filter: {df_filter.designed_sequence.nunique()} "
        f"unique designs kept ({n_drop} dropped)"
    )

if df_filter.empty:
    raise ValueError(
        f"All designs were removed by the pI<{PI_THRESHOLD} filter. "
        f"Your binders' pI ranges {df_result.isoelectric_point.min():.2f}–"
        f"{df_result.isoelectric_point.max():.2f}. Raise PI_THRESHOLD or set it "
        f"to None to disable the filter."
    )


def select(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    is_scaling = df.critic_name.str.contains(
        SCALING_CHECKPOINT_SUBSTRING, regex=False, na=False
    )
    iptm_proxy = df.distogram_iptm_proxy.where(
        ~df.is_antibody, df.cdr_distogram_iptm_proxy
    )

    df["iptm_score"] = df.iptm.where(~is_scaling)
    df["iptm_proxy_score"] = iptm_proxy.where(is_scaling)
    scores = df.groupby("designed_sequence", as_index=False).agg(
        iptm_score=("iptm_score", "mean"), iptm_proxy_score=("iptm_proxy_score", "mean")
    )
    scores["selection_score"] = 0.5 * scores.iptm_score.fillna(
        0
    ) + 0.5 * scores.iptm_proxy_score.fillna(0)

    return scores.nlargest(min(len(scores), TOP_N), "selection_score")


df_select = df_filter.groupby(["target_name", "binder_name"], dropna=False).apply(
    select, include_groups=False
)
df_select.to_parquet(save_dir / "selection.parquet", index=False)

print(f"[4] final selection: {len(df_select)} designs -> {save_dir / 'selection.parquet'}")
df_select

print(df_filter[["target_name", "binder_name"]].drop_duplicates())
# you'll see None, None
print("df_filter rows:", len(df_filter))   # should be 8

[1] rows collected from successful jobs: 48
    unique designed_sequence: 12


  0%|          | 0/48 [00:00<?, ?it/s]

[2] pI distribution: min=4.05 median=10.87 max=12.00
[3] pI filter DISABLED — keeping all designs
[4] final selection: 12 designs -> sweep/selection.parquet
  target_name binder_name
0        None        None
df_filter rows: 48


In [31]:
df_result[df_result.critic_name.eq("ESMFold2-Experimental-Cutoff2025")].drop(
    columns=["complex", "logits"]
)

,is_antibody,critic_name,batch_idx,designed_sequence,final_loss,iptm,distogram_iptm_proxy,cdr_distogram_iptm_proxy,target_name,target_sequence,binder_name,binder_sequence,use_scaling_critics,seed,batch_size,call_id,future,status,isoelectric_point
3,False,ESMFold2-Experimental-Cutoff2025,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.440806,0.206247,0.276735,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,MLLLLGAGGRAAAILRAAARAAAAALEAAAASGGGGLNIKIALLVA...,True,0,1,fc-01KTP7Z6K9RP8J3F5TFBT8TGMQ,FunctionCall.from_id('fc-01KTP7Z6K9RP8J3F5TFBT...,SUCCESS,11.711343
7,False,ESMFold2-Experimental-Cutoff2025,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,4.990142,0.564328,0.391981,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,PKKLILVIFLALLVALLLRKILRAAAAGADPEELLKLLELLLDLLE...,True,1,1,fc-01KTP7Z6QCV9NMNWE6E5XKN6KX,FunctionCall.from_id('fc-01KTP7Z6QCV9NMNWE6E5X...,SUCCESS,4.689297
11,False,ESMFold2-Experimental-Cutoff2025,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.457186,0.155141,0.244944,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,SPLAARAARILRLAIRLAERAERLFEEAARTGDERLLIIAALLLLL...,True,2,1,fc-01KTP7Z6VGHQRQBWMZ45A7NRCM,FunctionCall.from_id('fc-01KTP7Z6VGHQRQBWMZ45A...,SUCCESS,11.184636
15,False,ESMFold2-Experimental-Cutoff2025,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,4.878600,0.141573,0.236996,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,MNLSGLSKRQLILLAILVFIIMVATILKMAKENGLSDEEVIRLIKK...,True,3,1,fc-01KTP7Z6YWKK5QN053XB569BK4,FunctionCall.from_id('fc-01KTP7Z6YWKK5QN053XB5...,SUCCESS,8.141084
19,False,ESMFold2-Experimental-Cutoff2025,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.112158,0.150234,0.336345,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,MSARAIAAAAAAAAIIAALAAAIARLAARNGGGGEAARIAARLAGL...,True,4,1,fc-01KTP7Z71W4FR66N6NPVQ0WS19,FunctionCall.from_id('fc-01KTP7Z71W4FR66N6NPVQ...,SUCCESS,11.999968
23,False,ESMFold2-Experimental-Cutoff2025,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.605164,0.717188,0.684069,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,HLRVLVLARAAVEAVEAALELLRLALRLGGAGARILLRALKFLLLK...,True,5,1,fc-01KTP7Z76EFB0DT26VRKY297QA,FunctionCall.from_id('fc-01KTP7Z76EFB0DT26VRKY...,SUCCESS,11.999968
27,False,ESMFold2-Experimental-Cutoff2025,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.641458,0.182806,0.316475,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,MNLALVKTLITIAIIIKLIAIALVLKLARSGGLSAGAIITIITIIT...,True,6,1,fc-01KTP7Z7AM5PPFS4BEXEE4Y5F3,FunctionCall.from_id('fc-01KTP7Z7AM5PPFS4BEXEE...,SUCCESS,11.263674
31,False,ESMFold2-Experimental-Cutoff2025,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,4.982890,0.190757,0.233022,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,SLALAIAIIILAIIIVTILITAVLKKLAEAGGGDAGLLRLLRKLLK...,True,7,1,fc-01KTP7Z7EXYM6WX2RP4XDBJQGG,FunctionCall.from_id('fc-01KTP7Z7EXYM6WX2RP4XD...,SUCCESS,10.548527
35,False,ESMFold2-Experimental-Cutoff2025,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.028123,0.476753,0.388007,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,MGLGGKSVKTLVVVVVLLALLLIALIIRAAERRGVSPEVVLRAVGA...,True,8,1,fc-01KTP7Z7NATW6EPC0XCT8GF3FZ,FunctionCall.from_id('fc-01KTP7Z7NATW6EPC0XCT8...,SUCCESS,11.553847
39,False,ESMFold2-Experimental-Cutoff2025,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.248516,0.136444,0.316475,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,SLLLSKAELIAVLRRAARAAAEALEELLRRAGAGDPEAARLARPLA...,True,9,1,fc-01KTP7Z7SBZ2DVWNSDRMG21JEG,FunctionCall.from_id('fc-01KTP7Z7SBZ2DVWNSDRMG...,SUCCESS,10.528606


In [32]:
df_select

designed_sequence  \
target_name binder_name                                                         
NaN         NaN         6   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        0   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        1   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        7   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        8   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        3   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        4   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        11  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        2   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        9   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        5   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        10  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   

                            iptm_score  iptm_proxy_score  selection_score  
target_name binder_name                                                    
NaN         NaN         6     0.578221               NaN         0.289111  
                        0     0.516936               NaN         0.258468  
                        1     0.452853               NaN         0.226426  
                        7     0.360595               NaN         0.180298  
                        8     0.266728               NaN         0.133364  
                        3     0.242588               NaN         0.121294  
                        4     0.222075               NaN         0.111037  
                        11    0.201166               NaN         0.100583  
                        2     0.182794               NaN         0.091397  
                        9     0.170305               NaN         0.085152  
                        5     0.161911               NaN         0.080956  
                        10    0.159054               NaN         0.079527

In [13]:
from google.colab import files
files.download(str(save_dir / "selection.csv"))

NameError: name 'save_dir' is not defined

## Appendix

### Modal Primer

- **info: ephemeral vs deployment**  
  Ephemeral = temporary app from `modal run` or `app.run()`, stopped when the client exits. Deployment = persistent named app from `modal deploy`, reused and observable across runs. ([modal.com](https://modal.com/docs/guide/apps?utm_source=openai))

- **info: dashboard**  
  Generic dashboard/apps page: [https://modal.com/apps](https://modal.com/apps). Modal also prints app/deployment links during runs/deploys. ([modal.com](https://modal.com/docs/guide/apps?utm_source=openai))

- **cli: ephemeral run**  
  ```bash
  modal run path/to/app.py
  ```

- **cli: deploy/redeploy**  
  ```bash
  modal deploy path/to/app.py
  ```
  Running this on an existing app name redeploys a new version. ([modal.com](https://modal.com/docs/reference/cli/deploy?utm_source=openai))

- **local: ephemeral from Python**  
  ```python
  with modal.enable_output():
      with modal_app.run():
          result = local_modal_obj.remote(...)
  ```

- **local: call a deployment**  
  ```python
  Cls = modal.Cls.from_name("app-name", "ClassName")
  obj = Cls(...)
  result = obj.method.remote(...)
  ```
  `Cls.from_name` references a class from a deployed app lazily. ([modal.com](https://modal.com/docs/reference/modal.Cls?utm_source=openai))